In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import os
import json
import glob
import pprint as pp
import math
import shutil

from openai import OpenAI as OpenAIClient

from typing import List, Optional, Literal, Tuple, Final, Dict, Any
from pydantic import BaseModel, Field, conint, confloat, ConfigDict

from collections import defaultdict

from google.colab import drive

In [ ]:
drive.mount('/content/drive')

os.environ["OPENAI_API_KEY"] = "..."

DATA_DIR: Final = Path('/content/drive/MyDrive/Data/LLM_Climate_Comparison')
CORPORATE_DATA_DIR: Final = DATA_DIR / "corporate_data"
SOURCE_DATA_DIR: Final = CORPORATE_DATA_DIR / "source"
PERSIST_DIR = CORPORATE_DATA_DIR / "persist"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Baseline LLM Approach

## Define Structured Output

In [ ]:
# Note that it says SBTi below - this just means aligned with SBTi concepts
TargetHorizon = Literal["near_term", "long_term", "net_zero"]
MetricType = Literal["absolute", "intensity"]
Ambition = Literal["1.5C", "well_below_2C", "2C", "unspecified"]
Status = Literal["approved", "committed", "in_validation", "expired", "unknown"]
TargetType = Literal[
    "sbti_near_term",         # sbti-style, near-term reductions e.g. by 2030
    "sbti_net_zero",          # sbti-style, long-term ambitions e.g. by 2050
    "non_target_claim"        # other kind of climate targets - filter these out later
]

class Target(BaseModel):
  title: Optional[str] = Field(None, description="Human-friendly label if present in text")
  target_type: TargetType = Field(..., description="Classify each item.")
  horizon: TargetHorizon = Field(..., description="near_term / long_term / net_zero (SBTi framing)")
  metric_type: MetricType = Field(..., description="Absolute or intensity target")
  scopes_covered: List[Literal["S1", "S2", "S3"]] = Field(..., description="Which scopes this target covers")
  scope3_categories: Optional[List[conint(ge=1, le=15)]] = Field(
        None, description="If S3: list of GHGP category numbers 1–15, if specified"
  )
  ambition: Ambition = Field(..., description="Declared temperature alignment (if stated)")
  coverage_pct: Optional[confloat(ge=0, le=100)] = Field(
        None, description="Share of emissions covered by the target (company-wide or in-scope)"
  )

  base_year: Optional[int] = Field(None, description="Baseline year (e.g., 2019)")
  target_year: Optional[int] = Field(None, description="Completion year (e.g., 2030)")
  reduction_pct: Optional[confloat(ge=0, le=100)] = Field(
        None, description="Required % reduction vs base (for absolute) or intensity"
  )
  base_value: Optional[float] = Field(
        None, description="If explicitly given (e.g., baseline tCO2e or intensity)"
  )
  target_value: Optional[float] = Field(
        None, description="If explicitly given (e.g., target tCO2e or intensity)"
  )
  unit: Optional[str] = Field(
        None, description="Unit for base/target values (e.g., tCO2e, tCO2e/$, %, ktCO2e)"
  )

  status: Status = Field(..., description="SBTi status if mentioned")
  boundary: Optional[str] = Field(
        None, description="Organizational/operational boundary (e.g., company-wide, specific BU/geography)"
  )

  notes: Optional[str] = Field(None, description="Any qualifiers (market-based/location-based, exclusions, etc.)")
  sources: List[str] = Field(..., description="Doc names/URLs/node IDs where this target was found")

class ExtractedTargets(BaseModel):
    company: Optional[str] = None
    targets: List[Target] = Field(
        default_factory=list,
        description="All SBTi-style targets found in the context"
    )


## Establish System Prompt

In [ ]:
SYSTEM_PROMPT = """
You produce structured climate targets data using only your general world knowledge.
Output must be valid JSON matching the provided schema—no prose.

Rules:
- Populate the schema with all distinct climate targets you know; do not merge distinct targets.
- Do not invent specifics. If you are not confident a detail is correct, set it to null (or "unknown" where the schema requires).
- Do not reference any “provided context”, “documents”, “retrieved nodes”, or citations. Use sources = ["world_knowledge"] for every target.

Target typing & separation:
- horizon ∈ {near_term, long_term, net_zero}
- metric_type ∈ {absolute, intensity}
- ambition ∈ {1.5C, well_below_2C, 2C, unspecified}
- status ∈ {approved, committed, in_validation, expired, unknown}
- Create SEPARATE targets when any of these differ: base_year, target_year, scopes_covered, metric_type, ambition, status, boundary, reduction_pct.

Scopes:
- Do NOT default scopes. Only include a scope if you are confident it is part of that target.
- Set scopes_covered to [] unless you are confident S1/S2/S3 (or “Scope 1/2/3”) are covered for that target.
- If Scope 3 is covered, include scope3_categories (1–15) only if you are confident; else null.

Coverage & ambition:
- coverage_pct must be numeric only if known; else null.
- ambition only if explicitly named or unambiguously implied by standard terminology; else "unspecified".

Units & numerics:
- Percent reduction → unit = "%".
- Preserve numeric precision as written/known.
- If base_value/target_value are known, include unit (e.g., "tCO2e", "tCO2e/$"); else null.

Boundary & notes:
- boundary only if known; else null.
- notes: one short sentence of qualifiers; no speculation.

Empty result:
- If you do not know any relevant target data, output exactly:
```json
{"company": null, "targets": []}
"""


In [ ]:
QUERY_PROMPT = (
  "Using only your general world knowledge, populate the schema with this company's climate "
  "emissions targets as of disclosure year {disclosure_year} for {company}. "
  "Focus on SBTi-style target definitions (near-term and long-term/net-zero), "
  "and classify each target as absolute or intensity. Capture scope coverage (Scopes 1, 2, 3) "
  "only when you are confident it applies to that target. "
  #"Do not cite or refer to any provided context or documents. "
  #"Do not invent specifics. If any field is not known with confidence, set it to null "
  '(or "unknown" where required by the schema).'
)

## LLM Code

In [ ]:
client = OpenAIClient()

def run_structured(
  *,
  system_prompt: str,
  user_prompt: str,
  output_model,  # e.g. ExtractedTargets (Pydantic)
  model: str = "gpt-5-2025-08-07", # "gpt-5.2-2025-12-11", "gpt-5.1-2025-11-13",   #"gpt-5-2025-08-07",
  max_output_tokens: int = 16390,
  reasoning_effort: str = "high",  # "low" | "medium" | "high"
) -> ExtractedTargets:
  resp = client.responses.parse(
    model=model,
    reasoning={"effort": reasoning_effort},
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    text_format=output_model,
    max_output_tokens=max_output_tokens,
    )
  return resp.output_parsed

In [ ]:
def write_company_year_json(
  company_ticker: str,
  year: str,
  model_obj: ExtractedTargets
) -> Path:
  """ writes the json output for the QUERY_PROMPT
      response for a given company and year
  """

  output_persist_dir = PERSIST_DIR / "output_baseline_checks"
  output_persist_dir.mkdir(parents=True, exist_ok=True)

  schema_version = "v1"
  out_path = output_persist_dir / f"{company_ticker}.{year}.targets.{schema_version}.json"
  payload = model_obj.model_dump(mode="json")

  json_str = json.dumps(
    payload,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
    allow_nan=False,)

  out_path.write_text(json_str + "\n", encoding="utf-8")
  print(f"wrote {out_path}")

  return out_path

In [ ]:
## Test Run (single company, year)
company_ticker = "aapl"
year = "2024"

user_prompt = QUERY_PROMPT.format(company=company_ticker, disclosure_year=year)
#result = run_structured(system_prompt=SYSTEM_PROMPT,user_prompt=user_prompt, output_model=ExtractedTargets)
#print(result)

In [ ]:
def batch_run(
  years: List[str],
  company_tickers: List[str],
  model: str = "gpt-5-2025-08-07", # "gpt-5.2-2025-12-11", "gpt-5.1-2025-11-13",
  ):
  print("running model", model)
  for company_ticker in company_tickers:
    for year in years:
      company_ticker = company_ticker.lower()
      print("processing:", company_ticker, year)
      user_prompt = QUERY_PROMPT.format(company=company_ticker, disclosure_year=year)
      result = run_structured(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        output_model=ExtractedTargets,
        model=model,
      )
      pp.pprint(result)
      _ = write_company_year_json(company_ticker, year, result)

In [ ]:
company_ticker

'aapl'

# Evaluation - LLM as Judge

## Build Evaluation Framework

In [ ]:
FIELDS_TO_SCORE = [
  "ambition", "base_year", "horizon",
  "metric_type","reduction_pct",
  "scopes_covered", "target_year",
  "target_value", "unit"
]

EVAL_SYSTEM_PROMPT = """
You are an expert evaluator of structured climate targets (SBTi-style).
Your job: (1) ALIGN predicted targets to gold targets (1:1 matches), ignoring any target whose target_type = "non_target_claim";
(2) SCORE ONLY these fields: ambition, base_year, horizon, metric_type, reduction_pct, scopes_covered, target_year, target_value, unit
(3) Be robust to formatting differences, synonyms, and minor numeric/rounding differences.
(4) IMPORTANT: Perform both MATCHING and SCORING **using only the listed fields**; treat all other fields as unavailable.
(5) Use the rubric:
- EXACT = clearly the same value/meaning (case/format differences OK; synonyms OK; minor numeric diffs OK: ±0.5 abs OR ±1% relative)
- PARTIAL = meaning overlaps but is not fully correct (e.g., scopes partially overlap, or ambition “well_below_2C” vs “2C”)
- WRONG = materially different or missing when present in gold

Match policy:
- Create pairs between gold and predicted targets that represent the same real-world target.
- Prefer pairs that share the same target_type, horizon, and scope coverage.
- Each gold can match at most one prediction; each prediction can match at most one gold.

Return STRICT JSON:
{
  "matches": [
    {
      "gold_index": <int>,
      "pred_index": <int>,
      "field_scores": {
        "<field>": {"grade": "EXACT|PARTIAL|WRONG", "note": "<short reason>"}, ...
      }
    },
    ...
  ],
  "unmatched_gold": [<int>, ...],
  "unmatched_pred": [<int>, ...]
}
No extra keys. No commentary outside JSON.
"""

USER_TEMPLATE = """
Gold JSON for one document:
{gold}

Predicted JSON for the same document:
{pred}

Remember:
- Ignore targets where target_type == "non_target_claim" on BOTH sides.
- Only score these fields: {fields}.
- Return STRICT JSON as specified.
"""

In [ ]:
Grade = Literal["EXACT", "PARTIAL", "WRONG"]

class FieldScore(BaseModel):
  model_config = ConfigDict(extra="forbid")
  grade: Grade
  note: str

class FieldScores(BaseModel):
  # This forces exactly these keys (no more, no less)
  model_config = ConfigDict(extra="forbid")
  ambition: FieldScore
  base_year: FieldScore
  horizon: FieldScore
  metric_type: FieldScore
  reduction_pct: FieldScore
  scopes_covered: FieldScore
  target_year: FieldScore
  target_value: FieldScore
  unit: FieldScore

class Match(BaseModel):
  model_config = ConfigDict(extra="forbid")
  gold_index: int
  pred_index: int
  field_scores: FieldScores

class ScoreOutput(BaseModel):
  model_config = ConfigDict(extra="forbid")
  matches: List[Match]
  unmatched_gold: List[int]
  unmatched_pred: List[int]

def judge_align_and_score_openai(gold: dict, pred: dict, model="gpt-5-mini-2025-08-07") -> dict:
  client = OpenAIClient(api_key=os.getenv("OPENAI_API_KEY"))

  prompt = USER_TEMPLATE.format(
        gold=json.dumps(_project(gold), ensure_ascii=False),
        pred=json.dumps(_project(pred), ensure_ascii=False),
        fields=", ".join(FIELDS_TO_SCORE),
    )

  resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        text_format=ScoreOutput,
        reasoning={"effort": "high"},
    )

  parsed = getattr(resp, "output_parsed", None)

  if parsed is not None:
    return parsed.model_dump()

  txt = (getattr(resp, "output_text", None) or "").strip()
  try:
    return json.loads(txt)
  except:  # salvage JSON if model adds stray text
    i,j = txt.find("{"), txt.rfind("}")
    return json.loads(txt[i:j+1])

def _project(obj: dict) -> dict:
  """
    keep only certain fields for scoring
  """
  keep = set(FIELDS_TO_SCORE + ["target_type"])
  return {
    "company": obj.get("company"),
    "targets": [
      {k: t.get(k) for k in keep if k in t}
      for t in obj.get("targets", []) if t.get("target_type") != "non_target_claim"
    ],
  }

def _grade_to_score(g) -> float:
  """ convert a grade to a score
  """
  return 1.0 if g=="EXACT" else (0.5 if g=="PARTIAL" else 0.0)

def evaluate_dataset(test_cases: List[Dict[str, Any]]) -> Dict[str, Any]:
  """ evaluate a list of test cases.
  test_cases: [{"doc_name": str, "response": pred_json_str, "reference": gold_json_str}, ...]
  """
  per_doc = []
  tp = fp = fn = 0
  sums = {f:0.0 for f in FIELDS_TO_SCORE}
  cnts = {f:0 for f in FIELDS_TO_SCORE}

  for test_case in test_cases:
    doc = test_case["doc_name"]
    gold = json.loads(test_case["reference"])
    pred = json.loads(test_case["response"])
    judged = judge_align_and_score_openai(gold, pred)

    matches = judged.get("matches", [])
    ug = judged.get("unmatched_gold", [])
    up = judged.get("unmatched_pred", [])
    tp += len(matches)
    fp += len(up)
    fn += len(ug)

    field_acc = {f: None for f in FIELDS_TO_SCORE}
    for f in FIELDS_TO_SCORE:
      ok = [
        _grade_to_score(m.get("field_scores", {}).get(f, {}).get("grade", "WRONG"))
        for m in matches if f in m.get("field_scores", {})
      ]
      if ok:
        v = sum(ok)/len(ok)
        field_acc[f]=v
        sums[f]+=v
        cnts[f]+=1

    per_doc.append({"doc": doc, "tp": len(matches), "fp": len(up), "fn": len(ug), "field_accuracy": field_acc})

  micro_p = tp/(tp+fp) if tp+fp else 1.0
  micro_r = tp/(tp+fn) if tp+fn else 1.0
  micro_f1 = 2*micro_p*micro_r/(micro_p+micro_r) if micro_p+micro_r else 0.0
  hallucination_rate = fp/(tp+fp) if tp+fp else 0.0

  field_macro = {f:(sums[f]/cnts[f] if cnts[f] else None) for f in FIELDS_TO_SCORE}

  return {"aggregate":{
              "micro_precision": micro_p,
              "micro_recall": micro_r,
              "micro_f1": micro_f1,
              "hallucination_rate": hallucination_rate,  # <-- and return it
              "field_acc_macro": field_macro
          },
          "per_doc": per_doc}

In [ ]:
def get_json_pairs(
    company_ticker: str,
    year: str,
  ) -> Tuple[str, str, str]:
  output_dir = PERSIST_DIR / "output_baseline_checks"
  validation_dir = PERSIST_DIR / "validation_set"

  doc_name = f"{company_ticker}.{year}.targets.v1.json"

  test_json_path = output_dir / doc_name
  validation_json_path = validation_dir / doc_name

  test_json_str = json.dumps(json.loads(test_json_path.read_text()), ensure_ascii=False)
  validation_json_str = json.dumps(json.loads(validation_json_path.read_text()), ensure_ascii=False)

  return doc_name, test_json_str, validation_json_str

In [ ]:
def write_evaluation_json(
  filename: str,
  evaluation_dict: dict,
  output_persist_dir: Path
) -> Path:
  """ writes the json output for the evaluation
  """
  output_persist_dir.mkdir(parents=True, exist_ok=True)
  out_path = output_persist_dir / f"{filename}.json"

  json_str = json.dumps(
    evaluation_dict,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
    allow_nan=False,)

  out_path.write_text(json_str + "\n", encoding="utf-8")
  print(f"wrote {out_path}")

  return out_path

In [ ]:
def get_test_cases(
    company_tickers : List[str],
    years: List[str]
  ) -> List[dict]:
  test_cases = []
  for company_ticker in company_tickers:
    for year in years:
      company_ticker = company_ticker.lower()
      print(company_ticker, year)
      doc_name, test_json_str, validation_json_str = get_json_pairs(company_ticker, year)
      test_cases.append({
        "doc_name": doc_name,
        "response": test_json_str,
        "reference": validation_json_str,
      })
  return test_cases

## Count Eval Fields

In [ ]:
def count_fields(
    company_tickers,
    years,
    fields=FIELDS_TO_SCORE) -> dict:
  """
    company_year_pairs: iterable of (company_ticker, year)
    Uses your get_json() to load each file, then counts how many targets
    have each field present (non-None).
  """
  counts = {f: 0 for f in fields}

  for ticker in company_tickers:
    for year in years:
      ticker = ticker.lower()
      doc_name = f"{ticker}.{year}.targets.v1.json"
      validation_json_path = PERSIST_DIR / "validation_set" / doc_name
      data = json.loads(validation_json_path.read_text())
      #_, data = get_dict(ticker.lower(), year)
      for t in data.get("targets", []):
        for f in fields:
          if t.get(f) is not None:
            counts[f] += 1

  return counts

In [ ]:
company_info = {
    "NVDA": "NVIDIA Corporation",
    "MSFT": "Microsoft Corporation",
    "AAPL": "Apple Inc",
    "GOOGL": "Alphabet Inc",
    "AMZN": "Amazon.com Inc",
    "META": "Meta Platforms Inc",
    "TSLA": "Tesla Inc",
}

years = ["2023", "2024"]

In [ ]:
count_fields(company_info, years, fields=FIELDS_TO_SCORE)

{'ambition': 34,
 'base_year': 15,
 'horizon': 34,
 'metric_type': 34,
 'reduction_pct': 14,
 'scopes_covered': 34,
 'target_year': 29,
 'target_value': 4,
 'unit': 4}

# Batch Run

In [ ]:
def copy_dir_contents(src, dst):
  src, dst = Path(src), Path(dst)
  dst.mkdir(parents=True, exist_ok=True)

  for p in src.iterdir():
    target = dst / p.name
    if p.is_dir():
      shutil.copytree(p, target, dirs_exist_ok=True)
    else:
      shutil.copy2(p, target)

In [ ]:
models = {
     "gpt5" : "gpt-5-2025-08-07",
     "gpt5_1" : "gpt-5.1-2025-11-13",
     "gpt5_2" : "gpt-5.2-2025-12-11",
}

# set the particular model here
model_name = "gpt5"
model = models[model_name]

company_tickers = list(company_info.keys())

repeats = 5
reports = []

for i in range(repeats):
  output_dir = PERSIST_DIR / "output_baseline_checks"
  output_dir.mkdir(parents=True, exist_ok=True)
  shutil.rmtree(output_dir, ignore_errors=True)
  counter = i+1
  print("running report", counter, "for", model)
  batch_run(years, company_tickers)
  dest_dir = PERSIST_DIR / "llm_gen_raw" / model_name / f"run_{counter}"
  copy_dir_contents(output_dir, dest_dir)
  print("running eval", counter)
  test_cases = get_test_cases(company_tickers, years)
  report = evaluate_dataset(test_cases)
  print(report["aggregate"])
  reports.append(report)
  write_evaluation_json(f'full_report_{model_name}_{counter}',report, output_persist_dir = PERSIST_DIR / "evaluation_baseline")



running report 1 for gpt-5-2025-08-07
running model gpt-5-2025-08-07
processing: nvda 2023
ExtractedTargets(company=None, targets=[])
wrote /content/drive/MyDrive/Data/LLM_Climate_Comparison/corporate_data/persist/output_baseline_checks/nvda.2023.targets.v1.json
processing: nvda 2024
ExtractedTargets(company=None, targets=[])
wrote /content/drive/MyDrive/Data/LLM_Climate_Comparison/corporate_data/persist/output_baseline_checks/nvda.2024.targets.v1.json
processing: msft 2023
ExtractedTargets(company='Microsoft', targets=[Target(title='Reduce company-wide GHG emissions by more than half by 2030', target_type='sbti_near_term', horizon='near_term', metric_type='absolute', scopes_covered=['S1', 'S2', 'S3'], scope3_categories=None, ambition='unspecified', coverage_pct=None, base_year=None, target_year=2030, reduction_pct=None, base_value=None, target_value=None, unit=None, status='unknown', boundary='Company-wide value chain', notes='Microsoft stated it will cut total Scope 1–3 emissions by 

In [ ]:
for report in reports:
  print(json.dumps(report["aggregate"], indent=2))

{
  "micro_precision": 1.0,
  "micro_recall": 0.42857142857142855,
  "micro_f1": 0.6,
  "hallucination_rate": 0.0,
  "field_acc_macro": {
    "ambition": 0.7142857142857143,
    "base_year": 0.5238095238095238,
    "horizon": 0.7857142857142857,
    "metric_type": 1.0,
    "reduction_pct": 0.5714285714285714,
    "scopes_covered": 0.6428571428571429,
    "target_year": 0.8095238095238094,
    "target_value": 0.9285714285714286,
    "unit": 0.738095238095238
  }
}
{
  "micro_precision": 1.0,
  "micro_recall": 0.35714285714285715,
  "micro_f1": 0.5263157894736842,
  "hallucination_rate": 0.0,
  "field_acc_macro": {
    "ambition": 0.75,
    "base_year": 0.78125,
    "horizon": 0.9375,
    "metric_type": 1.0,
    "reduction_pct": 0.75,
    "scopes_covered": 0.6875,
    "target_year": 0.9375,
    "target_value": 0.875,
    "unit": 0.75
  }
}
{
  "micro_precision": 0.8461538461538461,
  "micro_recall": 0.39285714285714285,
  "micro_f1": 0.5365853658536585,
  "hallucination_rate": 0.15384615

In [ ]:
def mean_std_micro(results, ddof=1):
  keys = ("micro_precision", "micro_recall", "micro_f1", "hallucination_rate")
  out = {}
  for k in keys:
    xs = np.array([r["aggregate"][k] for r in results], dtype=float)
    out[k] = {
            "mean": float(xs.mean()),
            "std": float(xs.std(ddof=ddof)),
            "n": int(xs.size),
      }
  return out

In [ ]:
len(reports)

5

In [ ]:
mean_std_micro(reports, ddof=1)

{'micro_precision': {'mean': 0.947008547008547,
  'std': 0.07411789192710702,
  'n': 5},
 'micro_recall': {'mean': 0.34285714285714286,
  'std': 0.07405871911902757,
  'n': 5},
 'micro_f1': {'mean': 0.49906671755195503, 'std': 0.08153531577885126, 'n': 5},
 'hallucination_rate': {'mean': 0.05299145299145299,
  'std': 0.07411789192710701,
  'n': 5}}